# ICT-4202 | Artificial Intelligence and Machine Learning Laboratory
## Final Laboratory Examination — Set 02
**Islamic University, Kushtia, Bangladesh**  
Department of Information and Communication Technology

> **Notebook purpose:** Question-matched solutions with brief background theory, mathematical notation, readable Python, reproducible output, interpretation, and viva revision. This is a learning/reporting notebook, not a claim of official model answers.

| Question | Topic | Marks |
|:--|:--|--:|
| Q1 | Forward chaining with Cold | 25 |
| Q2 | Depth-limited search and IDDFS | 30 |
| Q3 | Simple linear regression | 35 |
| Q4 | Viva: facts/rules, cutoff, slope, R² | 10 |

**Execution:** Open in JupyterLab/VS Code, choose a Python kernel, then use **Run All**. All random experiments use a fixed seed. Markdown cells use MathJax (`$...$` and `$$...$$`) to render equations on GitHub/Jupyter.

**Source:** `AI_MLQuestionList.pdf` (uploaded exam sheet), pages 1; background concepts from the supplied ICT-4202 laboratory manual. The source questions govern specifications; where a question leaves a parameter open, our choice is explicitly labeled.

**Quick access - one file per numbered question:** [Q1 / A](Set_02_A.py) | [Q2 / B](Set_02_B.py) | [Q3 / C](Set_02_C.py) | [Q4 / D: Viva practice](Set_02_D.py).

Q1-Q3 contain the required practical work (including all subparts). Q4 is a viva, so its `.py` file is a revision/self-quiz tool, not an extra algorithm assignment. This notebook contains solutions, explanations, mathematical theory, and the Q4 viva questions.

---
## Q1. Knowledge-Based Agent (25 marks)
**Question specification**  
`Cloudy`; `Cloudy→Rain`; `Rain→WetGround`; `WetGround→Slippery`; `Slippery→DriveSlowly`; `Rain AND Cold→WearJacket`. Run forward chaining without Cold and with Cold. Print initial facts, rules fired, final knowledge, and explain `WearJacket`.

### Theory and mathematical background
A knowledge base (KB) stores **facts** and **rules**. In propositional forward chaining, rules have premises and a conclusion. For example:
$$P\land Q\Rightarrow R.$$
A rule fires when all its premises are already known. This is repeated until no new conclusion can be inferred (the *fixed point*).

For implication, $P\Rightarrow Q\equiv \neg P\lor Q$; it is false only when $P$ is true and $Q$ is false. Forward chaining for definite Horn-style rules is a data-driven form of logical inference.

In [1]:

def forward_chain(initial_facts, rules):
    """Data-driven inference: rules are (premise-set, conclusion) pairs."""
    known = set(initial_facts)
    trace = []
    while True:
        newly_added = False
        for premises, conclusion in rules:
            if premises.issubset(known) and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(premises), conclusion))
                newly_added = True
        if not newly_added:
            break
    return known, trace

def print_inference(label, initial, rules):
    known, trace = forward_chain(initial, rules)
    print(f"\n{label} | Initial facts: {sorted(initial)}")
    for i, (prem, conclusion) in enumerate(trace, 1):
        print(f"  {i}. {prem}  =>  {conclusion}")
    if not trace: print("  No new rule fired.")
    print('Final facts:', sorted(known))
    return known, trace

rules = [({'Cloudy'},'Rain'), ({'Rain'},'WetGround'),
         ({'WetGround'},'Slippery'), ({'Slippery'},'DriveSlowly'),
         ({'Rain','Cold'},'WearJacket')]
no_cold, t1 = print_inference('Case A: without Cold', {'Cloudy'}, rules)
with_cold, t2 = print_inference('Case B: with Cold', {'Cloudy','Cold'}, rules)
print('WearJacket without Cold?', 'WearJacket' in no_cold)
print('WearJacket with Cold?', 'WearJacket' in with_cold)
assert 'WearJacket' not in no_cold and 'WearJacket' in with_cold



Case A: without Cold | Initial facts: ['Cloudy']
  1. ['Cloudy']  =>  Rain
  2. ['Rain']  =>  WetGround
  3. ['WetGround']  =>  Slippery
  4. ['Slippery']  =>  DriveSlowly
Final facts: ['Cloudy', 'DriveSlowly', 'Rain', 'Slippery', 'WetGround']

Case B: with Cold | Initial facts: ['Cloudy', 'Cold']
  1. ['Cloudy']  =>  Rain
  2. ['Rain']  =>  WetGround
  3. ['WetGround']  =>  Slippery
  4. ['Slippery']  =>  DriveSlowly
  5. ['Cold', 'Rain']  =>  WearJacket
Final facts: ['Cloudy', 'Cold', 'DriveSlowly', 'Rain', 'Slippery', 'WearJacket', 'WetGround']
WearJacket without Cold? False
WearJacket with Cold? True


### Result interpretation and discussion
In the first run, `Rain` becomes known but `Cold` is not known, so the **conjunctive** rule does not fire. With `Cold` asserted, both premises are available and `WearJacket` is derived. Absence from a knowledge base is not automatically logical negation (open-world caution).

---
## Q2. Depth-Limited and Iterative-Deepening Search (30 marks)
**Question specification**  
`S→{A,B}, A→{C,D}, B→{E,F}, C→{H}, D→{G}, E→{I}, F→{J}`. Search S to G using DLS limits 2 and 3 and IDDFS; show trace, cutoff, and repeated shallow search.

### Theory and mathematical background
**Depth-Limited Search (DLS)** stops descending beyond an allowed depth $\ell$. A depth cutoff means the search might succeed with a larger limit; *failure* means no answer was found without such a cutoff.

**Iterative Deepening DFS (IDDFS)** performs DLS repeatedly at limits $0,1,2,\ldots$ until a goal is found:
$$IDDFS(\ell)=DLS(0),DLS(1),\dots,DLS(\ell).$$
It revisits shallow nodes, trading some repeated time for the memory efficiency of depth-first search. In a tree, DLS is commonly characterized by $O(b^\ell)$ time and $O(b\ell)$ space when sibling iterators are retained; IDDFS has roughly $O(b^d)$ time and $O(bd)$ space for a finite branching factor.

In [2]:
def depth_limited_search(graph, start, goal, limit):
    """Return path, status (success/cutoff/failure), expanded sequence."""
    expanded = []
    def visit(node, path, depth):
        expanded.append((node, depth))
        if node == goal: return path, 'success'
        if depth >= limit:
            eligible = [v for v in graph.get(node, []) if v not in path]
            return None, ('cutoff' if eligible else 'failure')
        had_cutoff = False
        for nxt in graph.get(node, []):
            if nxt in path: continue
            result, status = visit(nxt, path + [nxt], depth + 1)
            if status == 'success': return result, status
            if status == 'cutoff': had_cutoff = True
        return None, 'cutoff' if had_cutoff else 'failure'
    path, status = visit(start, [start], 0)
    return path, status, expanded

def iddfs(graph, start, goal, max_depth=10):
    iterations = []
    for limit in range(max_depth+1):
        path, status, expanded = depth_limited_search(graph, start, goal, limit)
        iterations.append((limit, path, status, expanded))
        if status == 'success': break
        if status == 'failure': break
    return path, iterations

def print_iddfs(graph, start, goal, max_depth=10):
    path, iterations = iddfs(graph, start, goal, max_depth)
    for limit, candidate, status, expanded in iterations:
        trace = ' -> '.join(node for node, depth in expanded)
        print(f'Limit={limit} | status={status:7} | expanded={trace} | path={candidate}')
    print(f'IDDFS path: {path}; root visits: {len(iterations)}')
    return path, iterations

graph = {'S':['A','B'],'A':['C','D'],'B':['E','F'],
         'C':['H'],'D':['G'],'E':['I'],'F':['J'],
         'G':[],'H':[],'I':[],'J':[]}
for limit in [2,3]:
    path,status,expanded = depth_limited_search(graph,'S','G',limit)
    print(f'DLS {limit}: status={status}; path={path}; expanded={expanded}')
path, iterations = print_iddfs(graph,'S','G',6)
assert path == ['S','A','D','G']


DLS 2: status=cutoff; path=None; expanded=[('S', 0), ('A', 1), ('C', 2), ('D', 2), ('B', 1), ('E', 2), ('F', 2)]
DLS 3: status=success; path=['S', 'A', 'D', 'G']; expanded=[('S', 0), ('A', 1), ('C', 2), ('H', 3), ('D', 2), ('G', 3)]
Limit=0 | status=cutoff  | expanded=S | path=None
Limit=1 | status=cutoff  | expanded=S -> A -> B | path=None
Limit=2 | status=cutoff  | expanded=S -> A -> C -> D -> B -> E -> F | path=None
Limit=3 | status=success | expanded=S -> A -> C -> H -> D -> G | path=['S', 'A', 'D', 'G']
IDDFS path: ['S', 'A', 'D', 'G']; root visits: 4


### Result interpretation and discussion
The goal is at depth 3 (`S→A→D→G`), so the limit-2 run is cut off before that node; limit 3 succeeds. IDDFS repeats nodes near the root in each iteration but needs memory proportional to depth rather than the breadth of the entire frontier.

---
## Q3. Linear Regression (35 marks)
**Question specification**  
Fit simple linear regression to `(1,4),(2,5),(3,8),(4,9),(5,11),(6,14)`; report slope, intercept, prediction at 7 hours, $R^2$, and plot. Change the score at 6 hours from 14 to 18 and refit.

### Theory and mathematical background
Simple linear regression estimates a numerical response from a predictor:
$$\hat y_i=\beta_0+\beta_1x_i.$$
The least-squares estimator minimizes
$$SSE=\sum_{i=1}^{n}\bigl(y_i-(\beta_0+\beta_1x_i)\bigr)^2,$$
with $\hat\beta_1=\frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}$ and $\hat\beta_0=\bar y-\hat\beta_1\bar x$.

The coefficient of determination is
$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$
An in-sample $R^2$ measures fit to the data used; test $R^2$ estimates performance on unseen data. With small test sets, the latter can be unstable or negative.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
X = np.array([1,2,3,4,5,6],dtype=float).reshape(-1,1)
y = np.array([4,5,8,9,11,14],dtype=float)
model = LinearRegression().fit(X,y)
y_changed = y.copy(); y_changed[-1] = 18
changed_model = LinearRegression().fit(X,y_changed)
for label, mdl, target in [('Original',model,y),('Modified last score',changed_model,y_changed)]:
    print(f'{label}: slope={mdl.coef_[0]:.4f}; intercept={mdl.intercept_:.4f}; '
          f'pred(7)={mdl.predict([[7]])[0]:.4f}; train R²={mdl.score(X,target):.4f}')
grid = np.linspace(1,7,80).reshape(-1,1)
plt.figure(figsize=(8,4.7))
plt.scatter(X[:,0],y,label='Original observations')
plt.scatter([6],[18],marker='x',s=95,label='Changed observation (6,18)')
plt.plot(grid[:,0],model.predict(grid),label='Original fitted line')
plt.plot(grid[:,0],changed_model.predict(grid),'--',label='Refitted line')
plt.xlabel('Study hours'); plt.ylabel('Score'); plt.title('Regression sensitivity to one target value')
plt.grid(alpha=0.2); plt.legend(); plt.tight_layout(); plt.show()


ModuleNotFoundError: No module named 'sklearn'

### Result interpretation and discussion
Increasing the last score changes both the least-squares slope/intercept and residual structure, so $R^2$ need not increase. With only six observations, a single changed point can strongly affect a fitted line; avoid interpreting the relationship as causal. Here $R^2$ is **training** performance only.

---
## Q4. Viva voce (10 marks)

| Question | Concise answer |
|:--|:--|
| What is forward chaining? | Repeated application of satisfied rules to asserted facts. |
| What is a knowledge base? | A set of encoded facts and rules. |
| What is cutoff? | Search reaches the specified limit before ruling out deeper goals. |
| Why IDDFS revisits the root? | It restarts DLS with an incremented limit. |
| What is regression slope? | Expected change in predicted response for a one-unit predictor change. |
| What is the intercept? | Predicted response at input zero. |
| What does R² measure? | Goodness of fit relative to predicting the response mean. |

| How is a fact different from a rule? | A fact is currently known, such as Cloudy. A rule is an implication such as Cloudy implies Rain. |
| What is depth-limited search (DLS)? | A depth-first search that does not expand any node beyond a specified depth limit. |
| What is IDDFS? | Iterative Deepening Depth-First Search runs DLS at increasing limits until a goal is found. |

**Review tip:** Explain the code by identifying the inputs, data structures, stopping conditions, evaluation metrics, and one limitation for each task.

---
### Submission checklist
- [ ] Execute every Python cell in sequence; check the saved output is current.
- [ ] Explain the main formula, trace, and interpretation for Q1, Q2, Q3.
- [ ] Record the original task settings, dataset, random seed, and actual test metrics.
- [ ] For WEKA tasks, run WEKA locally and attach the real result-buffer output/screenshot; notebook instructions **do not substitute for WEKA evidence**.
- [ ] Be ready to discuss one methodological limitation in viva.

**Integrity note:** Code and explanations are independently written for this study repository. Numeric results are calculated from the shown code and may vary with library versions where numerical or tie-breaking differences occur.